# Ejercicios propuestos: análisis experimental de algoritmos de búsqueda

Este notebook acompaña los ejercicios propuestos del capítulo 7. El propósito es contrastar, mediante experimentos reproducibles, el comportamiento de los algoritmos de búsqueda estudiados en la obra.

Cada sección conserva la misma estructura:

<div style="display:flex; justify-content:center; margin:1.2em 0;">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Elemento</th>
      <th>Descripción</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Implementación</td><td>Algoritmo escrito de forma directa para medir pasos y tiempo.</td></tr>
    <tr><td>Tabla</td><td>Resumen de complejidad temporal y espacial por escenario.</td></tr>
    <tr><td>Experimento</td><td>Ejecución para tamaños de arreglo en potencias de 10.</td></tr>
    <tr><td>Gráfica</td><td>Comparación entre tiempo experimental y función teórica del caso.</td></tr>
  </tbody>
</table>
</div>

Las gráficas siguen el estilo usado en los análisis experimentales del capítulo 2: fondo blanco, curva experimental azul, función teórica roja punteada, rejilla simple y resolución de 500 dpi.


In [ ]:
#@title Dependencias, algoritmos y utilidades de medición { display-mode: "form" }

from __future__ import annotations

import math
import random
import re
import unicodedata
import statistics
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 500
plt.rcParams["savefig.dpi"] = 500
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["savefig.edgecolor"] = "white"
Path("graficas").mkdir(parents=True, exist_ok=True)

N_VALUES = np.array([10**exponent for exponent in range(1, 6)], dtype=int)
TRIALS = 30
VALUE_EXPONENT = 7
VALUE_MAX = 10**VALUE_EXPONENT
RNG = random.Random(202607)


def busqueda_secuencial(arr, x):
    steps = 0
    for index, value in enumerate(arr):
        steps += 1
        if value == x:
            return index, steps
    return -1, steps


def busqueda_binaria(arr, x):
    a, b = 0, len(arr) - 1
    steps = 0
    while a <= b:
        steps += 1
        m = (a + b) // 2
        if arr[m] == x:
            return m, steps
        if arr[m] < x:
            a = m + 1
        else:
            b = m - 1
    return -1, steps


def busqueda_interpolacion(arr, x):
    a, b = 0, len(arr) - 1
    steps = 0
    while a <= b and arr[a] <= x <= arr[b]:
        steps += 1
        if arr[a] == arr[b]:
            return (a if arr[a] == x else -1), steps
        p = a + ((x - arr[a]) * (b - a)) // (arr[b] - arr[a])
        if p < a or p > b:
            return -1, steps
        if arr[p] == x:
            return p, steps
        if arr[p] < x:
            a = p + 1
        else:
            b = p - 1
    return -1, steps


def busqueda_saltos(arr, x):
    n = len(arr)
    step_size = max(1, int(math.sqrt(n)))
    prev = 0
    current = step_size
    steps = 0

    while prev < n and arr[min(current, n) - 1] < x:
        steps += 1
        prev = current
        current += step_size
        if prev >= n:
            return -1, steps

    for index in range(prev, min(current, n)):
        steps += 1
        if arr[index] == x:
            return index, steps
        if arr[index] > x:
            return -1, steps
    return -1, steps


def busqueda_exponencial(arr, x):
    n = len(arr)
    steps = 1
    if n == 0:
        return -1, 0
    if arr[0] == x:
        return 0, steps

    i = 1
    while i < n and arr[i] <= x:
        steps += 1
        if arr[i] == x:
            return i, steps
        i *= 2

    a = i // 2
    b = min(i, n - 1)
    found, binary_steps = busqueda_binaria(arr[a:b + 1], x)
    steps += binary_steps
    return (a + found if found != -1 else -1), steps


def busqueda_ternaria(arr, x):
    a, b = 0, len(arr) - 1
    steps = 0
    while a <= b:
        third = (b - a) // 3
        m1 = a + third
        m2 = b - third

        steps += 1
        if arr[m1] == x:
            return m1, steps
        steps += 1
        if arr[m2] == x:
            return m2, steps

        if x < arr[m1]:
            b = m1 - 1
        elif x > arr[m2]:
            a = m2 + 1
        else:
            a = m1 + 1
            b = m2 - 1
    return -1, steps


def sorted_unique_sample(n, rng=RNG):
    return sorted(rng.sample(range(1, VALUE_MAX + 1), int(n)))


def uniform_array(n):
    return list(range(1, int(n) + 1))


def interpolation_worst_array(n):
    n = int(n)
    if n < 2:
        return [1]
    return list(range(1, n)) + [VALUE_MAX]


def central_target(arr):
    return arr[len(arr) // 2]


def first_target(arr):
    return arr[0]


def random_existing_target(arr, rng=RNG):
    return arr[rng.randrange(len(arr))]


def missing_low_target(arr):
    return arr[0] - 1


def missing_high_target(arr):
    return arr[-1] + 1


def interpolation_worst_target(arr):
    return arr[-2] if len(arr) > 1 else arr[0]


@dataclass(frozen=True)
class CaseSpec:
    title: str
    notation: str
    array_builder: Callable[[int], list[int]]
    target_builder: Callable[[list[int]], int]
    theory: Callable[[np.ndarray], np.ndarray]
    filename_suffix: str


@dataclass(frozen=True)
class AlgorithmSpec:
    name: str
    function: Callable[[list[int], int], tuple[int, int]]
    cases: tuple[CaseSpec, CaseSpec, CaseSpec]


def median_time_and_steps(function, arr, target, trials=TRIALS):
    times = []
    steps = []
    for _ in range(trials):
        start = time.perf_counter_ns()
        _index, step_count = function(arr, target)
        elapsed = (time.perf_counter_ns() - start) / 1e9
        times.append(elapsed)
        steps.append(step_count)
    return statistics.median(times), statistics.median(steps)


def measure_case(spec, case):
    time_values = []
    step_values = []
    for n in N_VALUES:
        arr = case.array_builder(int(n))
        target = case.target_builder(arr)
        measured_time, measured_steps = median_time_and_steps(spec.function, arr, target)
        time_values.append(measured_time)
        step_values.append(measured_steps)
    return np.array(time_values, dtype=float), np.array(step_values, dtype=float)


def slugify(text):
    normalized = unicodedata.normalize("NFKD", text)
    ascii_text = normalized.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "_", ascii_text.lower()).strip("_")


def plot_case(spec, case, time_values, step_values):
    n = N_VALUES.astype(float)
    theory_values = case.theory(n).astype(float)

    fig, (ax_time, ax_steps) = plt.subplots(1, 2, figsize=(12, 4), facecolor="white")
    fig.suptitle(f"{spec.name} - {case.title}")

    ax_time.plot(N_VALUES, time_values, marker="o", linestyle="-", color="b", label="Función de complejidad experimental")
    ax_time.set_xscale("log", base=10)
    ax_time.set_xticks(N_VALUES)
    ax_time.set_xticklabels([rf"$10^{{{int(np.log10(value))}}}$" for value in N_VALUES])
    ax_time.set_xlabel("Tamaño del arreglo (n)")
    ax_time.set_ylabel("Tiempo promedio [s]", color="b")
    ax_time.tick_params(axis="y", colors="b")
    ax_time.grid(True)

    ax_time_theory = ax_time.twinx()
    ax_time_theory.plot(N_VALUES, theory_values, linestyle="dotted", color="red", label=f"Función de complejidad teórica {case.notation}")
    ax_time_theory.set_ylabel("Función teórica", color="red")
    ax_time_theory.tick_params(axis="y", colors="red")

    lines_a, labels_a = ax_time.get_legend_handles_labels()
    lines_b, labels_b = ax_time_theory.get_legend_handles_labels()
    ax_time.legend(lines_a + lines_b, labels_a + labels_b, loc="upper left")

    ax_steps.plot(N_VALUES, step_values, marker="o", linestyle="-", color="b", label="Pasos medidos")
    ax_steps.plot(N_VALUES, theory_values, linestyle="dotted", color="red", label=f"Pasos teóricos {case.notation}")
    ax_steps.set_xscale("log", base=10)
    ax_steps.set_yscale("log")
    ax_steps.set_xticks(N_VALUES)
    ax_steps.set_xticklabels([rf"$10^{{{int(np.log10(value))}}}$" for value in N_VALUES])
    ax_steps.set_xlabel("Tamaño del arreglo (n)")
    ax_steps.set_ylabel("Pasos")
    ax_steps.grid(True)
    ax_steps.legend(loc="upper left")

    plt.tight_layout()
    fig.savefig(f"graficas/{slugify(spec.name)}_{case.filename_suffix}.png", bbox_inches="tight", pad_inches=.05)
    plt.show()


def result_table(spec):
    rows = []
    for case in spec.cases:
        time_values, step_values = measure_case(spec, case)
        rows.append({
            "Caso": case.title,
            "n mínimo": int(N_VALUES[0]),
            "n máximo": int(N_VALUES[-1]),
            "Tiempo inicial [s]": f"{time_values[0]:.3e}",
            "Tiempo final [s]": f"{time_values[-1]:.3e}",
            "Pasos iniciales": f"{step_values[0]:.0f}",
            "Pasos finales": f"{step_values[-1]:.0f}",
            "Clase": case.notation,
        })
    return rows


def display_result_table(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)


def run_algorithm_analysis(spec):
    rows = []
    for case in spec.cases:
        time_values, step_values = measure_case(spec, case)
        rows.append({
            "Caso": case.title,
            "n mínimo": int(N_VALUES[0]),
            "n máximo": int(N_VALUES[-1]),
            "Tiempo inicial [s]": f"{time_values[0]:.3e}",
            "Tiempo final [s]": f"{time_values[-1]:.3e}",
            "Pasos iniciales": f"{step_values[0]:.0f}",
            "Pasos finales": f"{step_values[-1]:.0f}",
            "Clase": case.notation,
        })
        plot_case(spec, case, time_values, step_values)
    display_result_table(rows)


ONE = lambda n: np.ones_like(n, dtype=float)
LINEAR = lambda n: n
HALF_LINEAR = lambda n: (n + 1) / 2
LOG2 = lambda n: np.maximum(1, np.log2(np.maximum(n, 2)))
LOG3_TIMES_2 = lambda n: 2 * np.maximum(1, np.log(n) / np.log(3))
SQRT = lambda n: np.sqrt(n)
TWO_SQRT = lambda n: 2 * np.sqrt(n)
LOGLOG = lambda n: np.maximum(1, np.log2(np.log2(np.maximum(n, 4))))

ALGORITHMS = {
    "secuencial": AlgorithmSpec(
        name="Búsqueda secuencial",
        function=busqueda_secuencial,
        cases=(
            CaseSpec("Mejor caso", r"$T(n)\in\Omega(1)$", uniform_array, first_target, ONE, "mejor"),
            CaseSpec("Caso promedio", r"$T(n)\in\Theta(n)$", uniform_array, random_existing_target, HALF_LINEAR, "promedio"),
            CaseSpec("Peor caso", r"$T(n)\in O(n)$", uniform_array, missing_high_target, LINEAR, "peor"),
        ),
    ),
    "binaria": AlgorithmSpec(
        name="Búsqueda binaria",
        function=busqueda_binaria,
        cases=(
            CaseSpec("Mejor caso", r"$T(n)\in\Omega(1)$", uniform_array, central_target, ONE, "mejor"),
            CaseSpec("Caso promedio", r"$T(n)\in\Theta(\log_2 n)$", uniform_array, random_existing_target, LOG2, "promedio"),
            CaseSpec("Peor caso", r"$T(n)\in O(\log_2 n)$", uniform_array, missing_high_target, LOG2, "peor"),
        ),
    ),
    "interpolacion": AlgorithmSpec(
        name="Búsqueda por interpolación",
        function=busqueda_interpolacion,
        cases=(
            CaseSpec("Mejor caso", r"$T(n)\in\Omega(1)$", uniform_array, first_target, ONE, "mejor"),
            CaseSpec("Caso promedio", r"$T(n)\in\Theta(\log_2(\log_2 n))$", sorted_unique_sample, random_existing_target, LOGLOG, "promedio"),
            CaseSpec("Peor caso", r"$T(n)\in O(n)$", interpolation_worst_array, interpolation_worst_target, LINEAR, "peor"),
        ),
    ),
    "saltos": AlgorithmSpec(
        name="Búsqueda por saltos",
        function=busqueda_saltos,
        cases=(
            CaseSpec("Mejor caso", r"$T(n)\in\Omega(1)$", uniform_array, first_target, ONE, "mejor"),
            CaseSpec("Caso promedio", r"$T(n)\in\Theta(\sqrt{n})$", uniform_array, random_existing_target, SQRT, "promedio"),
            CaseSpec("Peor caso", r"$T(n)\in O(\sqrt{n})$", uniform_array, missing_high_target, TWO_SQRT, "peor"),
        ),
    ),
    "exponencial": AlgorithmSpec(
        name="Búsqueda exponencial",
        function=busqueda_exponencial,
        cases=(
            CaseSpec("Mejor caso", r"$T(n)\in\Omega(1)$", uniform_array, first_target, ONE, "mejor"),
            CaseSpec("Caso promedio", r"$T(n)\in\Theta(\log_2 n)$", uniform_array, random_existing_target, LOG2, "promedio"),
            CaseSpec("Peor caso", r"$T(n)\in O(\log_2 n)$", uniform_array, missing_high_target, LOG2, "peor"),
        ),
    ),
    "ternaria": AlgorithmSpec(
        name="Búsqueda ternaria",
        function=busqueda_ternaria,
        cases=(
            CaseSpec("Mejor caso", r"$T(n)\in\Omega(1)$", uniform_array, central_target, ONE, "mejor"),
            CaseSpec("Caso promedio", r"$T(n)\in\Theta(\log_3 n)$", uniform_array, random_existing_target, LOG3_TIMES_2, "promedio"),
            CaseSpec("Peor caso", r"$T(n)\in O(\log_3 n)$", uniform_array, missing_high_target, LOG3_TIMES_2, "peor"),
        ),
    ),
}


# Búsqueda secuencial

La búsqueda secuencial recorre el arreglo desde la primera posición hasta encontrar el objetivo o agotar todos los elementos. Su ventaja principal es que funciona sobre arreglos ordenados y desordenados; su costo crece con la cantidad de posiciones revisadas.

## Complejidad

<div style="display:flex; justify-content:center; margin:1.2em 0;">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Escenario</th>
      <th><i>T</i>(<i>n</i>)</th>
      <th><i>S</i>(<i>n</i>)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Mejor caso</td><td>$\Omega(1)$</td><td>$\Omega(1)$</td></tr>
    <tr><td>Caso promedio</td><td>$\Theta(n)$</td><td>$\Theta(1)$</td></tr>
    <tr><td>Peor caso</td><td>$O(n)$</td><td>$O(1)$</td></tr>
  </tbody>
</table>
</div>

## Criterio experimental

El mejor caso ubica el objetivo en la primera posición. El caso promedio elige una posición existente al azar. El peor caso busca un valor mayor que todos los elementos, lo que obliga a recorrer el arreglo completo.


In [ ]:
run_algorithm_analysis(ALGORITHMS["secuencial"])

# Búsqueda binaria

La búsqueda binaria requiere un arreglo ordenado. En cada iteración calcula el punto medio, compara el objetivo con ese valor y descarta la mitad que ya no puede contenerlo.

## Complejidad

<div style="display:flex; justify-content:center; margin:1.2em 0;">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Escenario</th>
      <th><i>T</i>(<i>n</i>)</th>
      <th><i>S</i>(<i>n</i>)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Mejor caso</td><td>$\Omega(1)$</td><td>$\Omega(1)$</td></tr>
    <tr><td>Caso promedio</td><td>$\Theta(\log_2 n)$</td><td>$\Theta(1)$</td></tr>
    <tr><td>Peor caso</td><td>$O(\log_2 n)$</td><td>$O(1)$</td></tr>
  </tbody>
</table>
</div>

## Criterio experimental

El mejor caso usa el elemento central. El caso promedio elige objetivos existentes al azar. El peor caso busca un valor fuera del rango superior del arreglo, manteniendo el patrón de divisiones hasta agotar el intervalo.


In [ ]:
run_algorithm_analysis(ALGORITHMS["binaria"])

# Búsqueda por interpolación

La búsqueda por interpolación estima la posición del objetivo a partir de la relación entre el valor buscado y los valores extremos del rango actual. Su comportamiento depende de la distribución de los datos.

## Complejidad

<div style="display:flex; justify-content:center; margin:1.2em 0;">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Escenario</th>
      <th><i>T</i>(<i>n</i>)</th>
      <th><i>S</i>(<i>n</i>)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Mejor caso</td><td>$\Omega(1)$</td><td>$\Omega(1)$</td></tr>
    <tr><td>Caso promedio</td><td>$\Theta(\log_2(\log_2 n))$</td><td>$\Theta(1)$</td></tr>
    <tr><td>Peor caso</td><td>$O(n)$</td><td>$O(1)$</td></tr>
  </tbody>
</table>
</div>

## Criterio experimental

El mejor caso usa un objetivo que la fórmula localiza de inmediato. El caso promedio usa muestras uniformes aleatorias dentro de un rango amplio. El peor caso usa un arreglo casi lineal con un último valor extremo; esa distribución sesga la estimación y obliga a avanzar de forma gradual.


In [ ]:
run_algorithm_analysis(ALGORITHMS["interpolacion"])

# Búsqueda por saltos

La búsqueda por saltos avanza en bloques de tamaño cercano a $\sqrt{n}$. Primero encuentra el bloque donde podría estar el objetivo y después realiza una búsqueda secuencial dentro de ese bloque.

## Complejidad

<div style="display:flex; justify-content:center; margin:1.2em 0;">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Escenario</th>
      <th><i>T</i>(<i>n</i>)</th>
      <th><i>S</i>(<i>n</i>)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Mejor caso</td><td>$\Omega(1)$</td><td>$\Omega(1)$</td></tr>
    <tr><td>Caso promedio</td><td>$\Theta(\sqrt{n})$</td><td>$\Theta(1)$</td></tr>
    <tr><td>Peor caso</td><td>$O(\sqrt{n})$</td><td>$O(1)$</td></tr>
  </tbody>
</table>
</div>

## Criterio experimental

El mejor caso encuentra el objetivo al inicio. El caso promedio selecciona un objetivo existente al azar. El peor caso usa un valor mayor que todos los elementos, forzando los saltos hasta el final.


In [ ]:
run_algorithm_analysis(ALGORITHMS["saltos"])

# Búsqueda exponencial

La búsqueda exponencial crece con índices $1, 2, 4, 8, \ldots$ hasta acotar el rango donde puede estar el objetivo. Luego aplica búsqueda binaria dentro de ese rango.

## Complejidad

<div style="display:flex; justify-content:center; margin:1.2em 0;">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Escenario</th>
      <th><i>T</i>(<i>n</i>)</th>
      <th><i>S</i>(<i>n</i>)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Mejor caso</td><td>$\Omega(1)$</td><td>$\Omega(1)$</td></tr>
    <tr><td>Caso promedio</td><td>$\Theta(\log_2 n)$</td><td>$\Theta(1)$</td></tr>
    <tr><td>Peor caso</td><td>$O(\log_2 n)$</td><td>$O(1)$</td></tr>
  </tbody>
</table>
</div>

## Criterio experimental

El mejor caso encuentra el objetivo en la primera posición. El caso promedio elige objetivos existentes al azar. El peor caso busca un valor superior al máximo del arreglo, lo que activa la fase exponencial y la fase binaria final.


In [ ]:
run_algorithm_analysis(ALGORITHMS["exponencial"])

# Búsqueda ternaria

La búsqueda ternaria divide el rango de búsqueda en tres segmentos mediante dos puntos medios. En cada iteración puede descartar uno o dos segmentos según la comparación con el objetivo.

## Complejidad

<div style="display:flex; justify-content:center; margin:1.2em 0;">
<table style="margin: 0 auto; text-align: center;">
  <thead>
    <tr>
      <th>Escenario</th>
      <th><i>T</i>(<i>n</i>)</th>
      <th><i>S</i>(<i>n</i>)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Mejor caso</td><td>$\Omega(1)$</td><td>$\Omega(1)$</td></tr>
    <tr><td>Caso promedio</td><td>$\Theta(\log_3 n)$</td><td>$\Theta(\log_3 n)$</td></tr>
    <tr><td>Peor caso</td><td>$O(\log_3 n)$</td><td>$O(\log_3 n)$</td></tr>
  </tbody>
</table>
</div>

## Criterio experimental

El mejor caso usa una posición central. El caso promedio elige objetivos existentes al azar. El peor caso usa un valor mayor que todos los elementos y obliga a reducir el rango hasta agotarlo. En el análisis espacial se considera la formulación recursiva del algoritmo, cuya pila de llamadas crece con la profundidad de división del rango.


In [ ]:
run_algorithm_analysis(ALGORITHMS["ternaria"])

# Lectura de los resultados

Las tablas finales de cada sección muestran los pasos medidos para el tamaño más pequeño y el más grande. La gráfica de la izquierda muestra tiempo real, que puede variar por la máquina, el sistema operativo y la carga del entorno. La gráfica de la derecha muestra pasos del algoritmo, por lo que suele reflejar con mayor claridad la forma de la función teórica.

Para ejercicios de análisis, conviene comparar primero la curva de pasos con la función teórica y después usar los tiempos como evidencia experimental complementaria.
